# Notebook 7 — Production Deployment

**Phase 4 of 6 · Packaging → Cloud Deploy → CI/CD**

---

In this notebook we take everything we built — the XGBoost RUL model, the FastAPI inference server, and the pre-computed results — and ship the API to a **live public URL**.

### What we cover

| Step | Topic |
|------|-------|
| 1 | Review the full local FastAPI service (`src/api.py`) |
| 2 | Create a *lite* deployment API (XGBoost only — fits Render free tier) |
| 3 | Write minimal deployment dependencies (`requirements-deploy.txt`) |
| 4 | Containerize with **Docker** (`Dockerfile`) |
| 5 | Configure **Render** for one-click cloud deploy (`render.yaml`) |
| 6 | Automate with **GitHub Actions** — every push to `main` redeploys |
| 7 | Test the live endpoint from this notebook |

### Why a *lite* API?

Our full `src/api.py` imports TensorFlow (LSTM autoencoder) which adds ~500 MB of Python packages.  
Render's free tier has a 512 MB RAM limit — TensorFlow alone would exceed it.  
The lite version keeps **XGBoost** (our primary RUL model) and drops TensorFlow, keeping the image under 300 MB.  
The full API with SHAP + LSTM anomaly scoring runs locally via `uvicorn src.api:app`.

## System Architecture

```
┌──────────────────────────────────────────────────────────────────┐
│                    TURBOFAN ANOMALY DETECTION                    │
└──────────────────────────────────────────────────────────────────┘

  Sensor stream (30-cycle rolling window)
         │
         ▼
  ┌─────────────┐   POST /predict   ┌──────────────────────────────┐
  │  Client App │ ───────────────►  │   FastAPI on Render (cloud)  │
  │  (requests/ │ ◄───────────────  │   • XGBoost RUL regression   │
  │  Streamlit) │    JSON response  │   • Risk level (HIGH / SAFE) │
  └─────────────┘                   └──────────────────────────────┘
                                              ▲
                                    GitHub push to main
                                              │
                                   ┌──────────────────┐
                                   │  GitHub Actions  │
                                   │  1. smoke test   │
                                   │  2. trigger      │
                                   │     Render hook  │
                                   └──────────────────┘
```

The **defense / DoD framing**: any platform receiving telemetry from turbofan engines (aircraft, ships, ground vehicles) can call `/predict` with a feature vector and get a real-time RUL estimate + risk classification — exactly what a Palantir or Booz Allen operational dashboard would consume.

In [ ]:
from pathlib import Path

ROOT = Path('..').resolve()
print(f'Project root : {ROOT}')
print(f'Models       : {list((ROOT / "models").iterdir())}')
print(f'Results      : {list((ROOT / "results").iterdir())}')

---
## Step 1 — Review the Full Local API (`src/api.py`)

The full inference server exposes:
- **`POST /predict`** — accepts a 110-element feature vector **and** a 30×11 sensor window.  
  Returns: RUL estimate, risk level, top-5 SHAP drivers, LSTM anomaly score, anomaly flag.
- **`GET /health`** — liveness probe used by Docker and Render.

This version is what runs locally (`uvicorn src.api:app --reload`). It requires TensorFlow for the LSTM autoencoder, so it is **not** deployed to the cloud free tier.

In [ ]:
print(open(ROOT / 'src/api.py').read())

---
## Step 2 — The Lite Deployment API (`src/api_lite.py`)

We already created `src/api_lite.py`. Here is what it does differently from the full API:

| Feature | Full API | Lite API |
|---------|----------|----------|
| XGBoost RUL prediction | ✅ | ✅ |
| Risk level (HIGH / SAFE) | ✅ | ✅ |
| Cycles to alert | ✅ | ✅ |
| SHAP top-5 drivers | ✅ | ❌ (local only) |
| LSTM anomaly score | ✅ | ❌ (local only) |
| TensorFlow required | Yes | **No** |
| Estimated Docker image | ~700 MB | **~280 MB** |
| Render free tier | ❌ Too large | ✅ Fits |

The input is simpler too — the client only needs to send the **110-element feature vector** (not the raw sensor window).

In [ ]:
print(open(ROOT / 'src/api_lite.py').read())

---
## Step 3 — Deployment Dependencies (`requirements-deploy.txt`)

The full `requirements.txt` (generated from our training environment) lists ~80 packages including TensorFlow, SHAP, Evidently, and Jupyter.  
For deployment we only need 4:

| Package | Purpose |
|---------|----------|
| `fastapi` | HTTP framework that defines our API |
| `uvicorn[standard]` | ASGI server that runs FastAPI in production |
| `xgboost` | Loads the trained model and runs predictions |
| `numpy` | Array operations on the feature vector |

Pinning exact versions ensures the deployed image is **reproducible** — the same packages that passed our tests will run in production.

In [ ]:
print(open(ROOT / 'requirements-deploy.txt').read())

---
## Step 4 — Containerize with Docker

**What is Docker?**  
Docker packages an application and *all* its dependencies into a **container image** — a self-contained unit that runs identically on any machine (your laptop, Render's servers, AWS, etc.).

**Our Dockerfile uses a two-stage build:**

1. **Stage 1 (builder)** — a full Python image that installs packages.  
   We do this in a separate stage so the intermediate build artifacts don't pollute the final image.

2. **Stage 2 (runtime)** — a slim Python image that only copies the installed packages and our app files.

This pattern keeps the final image smaller than a naive single-stage build.

**Files copied into the image:**
- `src/api_lite.py` — the inference server
- `models/xgb_rul.json` — trained XGBoost model (1.2 MB JSON)
- `data/processed/feature_cols.pkl` — feature column names (needed for validation)

In [ ]:
print(open(ROOT / 'Dockerfile').read())

In [ ]:
# These commands build and run the Docker image locally.
# Run them in your terminal from the project root (requires Docker Desktop).

docker_commands = """
# Build the image (run from project root)
docker build -t turbofan-api .

# Run the container locally on port 8000
docker run -p 8000:8000 turbofan-api

# Test it (in a second terminal)
curl http://localhost:8000/health
"""
print(docker_commands)

---
## Step 5 — Deploy to Render

**Render** is a cloud platform with a free tier for web services.  
It reads `render.yaml` from the repository root and handles building, running, and scaling automatically.

**How Render deploys our service:**
1. Render detects `render.yaml` in the repo.
2. Runs `buildCommand` → `pip install -r requirements-deploy.txt`
3. Starts the server with `startCommand` → `uvicorn src.api_lite:app ...`
4. Polls `healthCheckPath` (`/health`) to confirm the service is live.
5. Assigns a public URL: `https://turbofan-anomaly-api.onrender.com`

> **Note:** Free tier services spin down after 15 minutes of inactivity. The first request after inactivity has a ~30-second cold-start delay. This is normal for a free demo.

In [ ]:
print(open(ROOT / 'render.yaml').read())

### Render Deployment Walkthrough

Follow these steps **once** to get your public URL:

1. Go to [render.com](https://render.com) and sign up with your GitHub account.
2. Click **New → Web Service**.
3. Select your repository `VladaSky/turbofan-anomaly-detection`.
4. Render will auto-detect `render.yaml` and pre-fill the settings.
5. Confirm:
   - **Runtime**: Python
   - **Build Command**: `pip install -r requirements-deploy.txt`
   - **Start Command**: `uvicorn src.api_lite:app --host 0.0.0.0 --port $PORT`
   - **Plan**: Free
6. Click **Create Web Service**.
7. Watch the build logs — it takes ~3 minutes the first time.
8. Once live, your URL appears at the top: `https://YOUR-APP-NAME.onrender.com`

#### Get the Deploy Hook (needed for GitHub Actions)

1. In Render dashboard → your service → **Settings** → **Deploy Hook**.
2. Copy the hook URL.
3. In GitHub → your repo → **Settings** → **Secrets and variables** → **Actions** → **New repository secret**.
4. Name: `RENDER_DEPLOY_HOOK`, Value: paste the URL.
5. Now every push to `main` will trigger a redeploy automatically.

---
## Step 6 — GitHub Actions CI/CD

**What is CI/CD?**
- **CI (Continuous Integration)** — automatically test code every time you push.
- **CD (Continuous Deployment)** — automatically deploy if the tests pass.

Our workflow (`.github/workflows/deploy.yml`) runs on every push to `main` and does two things:

**Job 1 — Test:**
1. Checks out the code on a clean Ubuntu VM.
2. Installs our 4 deployment dependencies.
3. Starts the API server (`uvicorn`) in the background.
4. Hits `/health` with `curl` and asserts the response is `{"status": "ok"}`.
5. If the check fails, the deploy is blocked.

**Job 2 — Deploy (only runs if Job 1 passed):**
1. Sends a `POST` request to the Render deploy hook URL (stored as a GitHub secret).
2. Render pulls the latest commit from `main` and rebuilds the service.

This means: **if it passes the health check on GitHub's servers, it ships to Render.**

In [ ]:
print(open(ROOT / '.github/workflows/deploy.yml').read())

---
## Step 7 — Test the Live API

We'll send a real prediction request to the deployed endpoint using engine 85's last window —
the same engine we analyzed in Notebooks 4 and 6.

The 110-element feature vector contains:
- 11 raw sensor values (normalized per engine)
- 99 rolling statistics (5/10/30-cycle mean, std, rate-of-change per sensor)

The API returns the predicted RUL and risk classification in ~10ms.

In [ ]:
import numpy as np

# Load the precomputed test data
X_feat   = np.load(ROOT / 'data/processed/X_features.npy')
eng_ids  = np.load(ROOT / 'data/processed/engine_ids_flat.npy')

# Engine 85, last available window (closest to failure)
last_idx = np.where(eng_ids == 85)[0][-1]
feature_vec = X_feat[last_idx].tolist()

print(f'Using window index : {last_idx}')
print(f'Feature vector dim : {len(feature_vec)}')
print(f'First 5 values     : {[round(v,4) for v in feature_vec[:5]]}')

payload = {
    'engine_id':      'engine_85',
    'feature_vector': feature_vec
}

In [ ]:
import requests

# ── Test the LOCAL API ──────────────────────────────────────────────────────
# Start it first in a terminal:
#   uvicorn src.api_lite:app --reload --host 0.0.0.0 --port 8000

LOCAL_URL = 'http://localhost:8000'

try:
    health = requests.get(f'{LOCAL_URL}/health', timeout=3)
    print('Health:', health.json())

    resp = requests.post(f'{LOCAL_URL}/predict', json=payload, timeout=5)
    print('\nPrediction response:')
    for k, v in resp.json().items():
        print(f'  {k:20s}: {v}')
except requests.exceptions.ConnectionError:
    print('Local API is not running.')
    print('Start it with:  uvicorn src.api_lite:app --reload')

In [ ]:
# ── Test the LIVE Render endpoint ───────────────────────────────────────────
# Replace this with your actual Render URL after deploying.

RENDER_URL = 'https://turbofan-anomaly-api.onrender.com'  # update after deploy

try:
    print(f'Calling {RENDER_URL}/health ...')
    health = requests.get(f'{RENDER_URL}/health', timeout=35)  # allow cold start
    print('Health:', health.json())

    print(f'\nCalling {RENDER_URL}/predict ...')
    resp = requests.post(f'{RENDER_URL}/predict', json=payload, timeout=35)
    print('Prediction response:')
    for k, v in resp.json().items():
        print(f'  {k:20s}: {v}')

except requests.exceptions.ConnectionError as e:
    print(f'Could not reach {RENDER_URL}')
    print('Make sure you have deployed to Render and updated RENDER_URL above.')
except requests.exceptions.Timeout:
    print('Request timed out — the service may still be cold-starting (up to 60s).')
    print('Try running this cell again in 30 seconds.')

---
## Summary — Deployment Checklist

| Task | File | Status |
|------|------|--------|
| Lite API (XGBoost only) | `src/api_lite.py` | ✅ Created |
| Deployment requirements | `requirements-deploy.txt` | ✅ Created |
| Docker container | `Dockerfile` | ✅ Created |
| Render config | `render.yaml` | ✅ Created |
| CI/CD pipeline | `.github/workflows/deploy.yml` | ✅ Created |
| Deploy to Render | render.com | ⬜ Manual step |
| Add `RENDER_DEPLOY_HOOK` secret | GitHub Settings | ⬜ Manual step |
| Update `RENDER_URL` in this notebook | Cell above | ⬜ After deploy |

### What we achieved

- Our turbofan anomaly detection model is now accessible from **any device in the world** via a REST API.
- Every push to `main` automatically tests and redeploys the service — **no manual intervention** needed.
- The complete inference pipeline (EDA → preprocessing → modeling → evaluation → RUL regression → API → deployment) is end-to-end documented in 7 notebooks.

### Next: Streamlit Dashboard

Notebook 8 will build an operational dashboard that:
- Shows a **fleet health grid** (all engines, color-coded by risk level)
- Plots **live RUL trends** per engine
- Renders a **SHAP waterfall** explaining why a specific engine is flagged
- Provides **maintenance scheduling recommendations**

The dashboard will call the deployed Render API — connecting everything into a single operational system.